# Lesson 5 — Building the RAG Assistant

### AI Study Buddy · Lesson 5 of 8

### This is the last notebook lesson. Next time we build the real project.

Everything meets today.

- Lesson 2 gave us **grounding**: put the text in the prompt, and the answer is anchored
  to it. But we could only paste everything, and everything doesn't scale.
- Sessions 3 and 4 gave us **retrieval**: find the handful of chunks that are relevant to
  a specific question.

Put retrieval in front of grounding and you have **RAG** — Retrieval-Augmented Generation.
That's the whole technique. It's genuinely that simple, and it is the most widely deployed
AI architecture in the world right now.

## TODAY:

- **Part A:** retrieve, assemble, and *read the actual prompt being sent*
- **Part B:** `answer_question()` — with sources
- **Part C:** the refusal, which is the point of all this
- **Part D:** the two dials, and the one-line provider swap

In [ ]:
# imports

import os
from pathlib import Path
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
# Setup

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
print(f"✅ Key loaded, begins {api_key[:11]}" if api_key else "❌ No key found - check your .env")

CHAT_MODEL = "openai:gpt-5.4-mini"
DOCUMENTS_DIR = "documents"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
K = 4                      # how many chunks to retrieve per question
EMBEDDING_MODEL = "text-embedding-3-small"

In [ ]:
# Rebuild everything from sessions 3 and 4. This is the last time we copy it -
# next session it becomes a real file we import.

def load_documents(folder):
    """Read every .md file in a folder and return them as LangChain Documents."""
    return [
        Document(page_content=path.read_text(encoding="utf-8"), metadata={"filename": path.name})
        for path in sorted(Path(folder).glob("**/*.md"))
    ]


documents = load_documents(DOCUMENTS_DIR)
chunks = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
).split_documents(documents)

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
Chroma(embedding_function=embeddings).delete_collection()   # start fresh if you re-run this cell
vectorstore = Chroma.from_documents(chunks, embedding=embeddings)

model = init_chat_model(CHAT_MODEL)

print(f"{len(documents)} documents → {len(chunks)} chunks → ready")

## PART A: Assemble the prompt, then read it

Retrieval gives us a list of `Document` objects. The model wants text. So we join them.

In [ ]:
# Step 1: retrieve

question = "Why does every flight carry two trackers?"
found = vectorstore.similarity_search(question, k=K)

f"Found {len(found)} chunks"

In [ ]:
# Step 2: join their text into one string.
#
# The separator matters more than it looks. Without a clear break between chunks, the
# model can read the end of one and the start of the next as a single sentence and
# invent a connection that isn't there.

context = "\n\n---\n\n".join(doc.page_content for doc in found)

print(context[:600])

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 <code>"x".join(...)</code> again</h3>
<p style="color:#06c; margin-bottom:0;">
JavaScript: <code>docs.map(d => d.pageContent).join("\\n\\n---\\n\\n")</code><br/>
Python: <code>"\\n\\n---\\n\\n".join(doc.page_content for doc in found)</code><br/><br/>
Same operation, glue written first. The bit inside the brackets is a
<b>generator expression</b> — a list comprehension without the square brackets. It produces
values one at a time instead of building a whole list first, which is exactly what
<code>join</code> wants.
</p>
</div>

In [ ]:
# Step 3: the system prompt. Same shape as Lesson 2 - except now {context} gets
# four relevant chunks instead of every document we own.

SYSTEM_TEMPLATE = """
You are a study assistant. You answer questions using only the notes provided below.

NOTES:
{context}
"""

system_prompt = SYSTEM_TEMPLATE.format(context=context)

### Now read the whole thing

Before we send it, look at exactly what the model is about to receive. This is the single
most useful debugging habit in the whole of this technique — almost every "why did it say
that?" is answered by reading the prompt.

In [ ]:
print(system_prompt)
print("=" * 70)
print("HUMAN:", question)

That's it. That's RAG, in full, with nothing hidden.

Four paragraphs picked out of a pile by similarity search, pasted above the question. No
model was retrained. No AI learned anything. We just did a good job of deciding what to
paste.

In [ ]:
# Step 4: ask

response = model.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=question),
])

print(response.text)

## PART B: `answer_question()` — with sources

Let's wrap the whole thing up, and return the sources alongside the answer.

Returning the sources is not decoration. It's what makes the answer **checkable** — the
user can go and read the original and decide for themselves whether they believe it.

In [ ]:
def answer_question(question, k=K):
    """
    Answer a question using only the retrieved notes.

    Returns a tuple: (answer, list of filenames used).

    We return the sources as well as the answer so the user can verify it. An answer you
    can't trace back to a document is exactly the thing Lesson 2 taught us not to trust -
    even when it happens to be right.
    """
    found = vectorstore.similarity_search(question, k=k)
    context = "\n\n---\n\n".join(doc.page_content for doc in found)

    response = model.invoke([
        SystemMessage(content=SYSTEM_TEMPLATE.format(context=context)),
        HumanMessage(content=question),
    ])

    # A set removes duplicates - four chunks often come from the same two files.
    # sorted() turns it back into a predictable order so the output doesn't jump around.
    sources = sorted({doc.metadata["filename"] for doc in found})
    return response.text, sources

In [ ]:
answer, sources = answer_question("Which flight holds the altitude record, and what made it successful?")

print(answer)
print()
print("Sources:", ", ".join(sources))

In [ ]:
# A question whose answer is spread across two different files

answer, sources = answer_question("What went wrong on Flight 4, and what rule came out of it?")

print(answer)
print()
print("Sources:", ", ".join(sources))

## PART C: The refusal

Now the part that makes this trustworthy rather than merely clever.

Remember the end of Lesson 4: **a vector store always returns k chunks**, even when
nothing is relevant. Ask about football and you get four paragraphs about weather balloons.

So what happens?

In [ ]:
answer, sources = answer_question("Who won the 2018 World Cup?")

print(answer)
print()
print("Sources:", ", ".join(sources))

### Whatever it said, look at our system prompt again

```
You are a study assistant. You answer questions using only the notes provided below.
```

We told it to use only the notes. We never told it what to do when the notes don't contain
the answer. So it does whatever seems most natural — which, for a machine that predicts
plausible text, is usually to produce an answer anyway.

The fix is one sentence.

In [ ]:
STRICT_TEMPLATE = """
You are a study assistant. You answer questions about the user's own notes.

Use ONLY the notes below. Do not use anything you know from anywhere else.
If the notes do not contain the answer, say exactly: "That isn't in your documents."
Do not guess. Do not fill gaps. A short answer that is definitely correct is much
better than a long one that might not be.

NOTES:
{context}
"""

In [ ]:
def answer_question(question, k=K, template=STRICT_TEMPLATE):
    """
    Answer a question using only the retrieved notes, refusing when they don't cover it.

    template - swap this to compare different instructions. Making it a parameter rather
               than hardcoding it is what lets us A/B test prompts in the cells below,
               which is the only honest way to tell whether a prompt change helped.

    Returns (answer, sources).
    """
    found = vectorstore.similarity_search(question, k=k)
    context = "\n\n---\n\n".join(doc.page_content for doc in found)

    response = model.invoke([
        SystemMessage(content=template.format(context=context)),
        HumanMessage(content=question),
    ])
    return response.text, sorted({doc.metadata["filename"] for doc in found})

In [ ]:
# The same question that just got an invented answer

answer, sources = answer_question("Who won the 2018 World Cup?")
print(answer)

In [ ]:
# And check we didn't break the good behaviour - a question it SHOULD answer

answer, sources = answer_question("What is the maximum payload mass?")
print(answer)
print("Sources:", ", ".join(sources))

<div style="border-left: 6px solid #181; background: #f3fbf3; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#181; margin-top:0;">🌍 Where you'll see this in the real world</h3>
<p style="color:#181; margin-bottom:0;">
That refusal is the entire commercial value of RAG.<br/><br/>
A support chatbot that invents a refund policy costs a company money and trust. One that
says "I can't find that — let me pass you to a person" is useful. The difference between
those two products is a paragraph of English in a system prompt, plus the retrieval that
makes the paragraph honest.<br/><br/>
It also isn't a guarantee. A system prompt is a strong instruction, not a law of physics —
you will find questions where it answers anyway. Reducing that rate is an active research
problem, and the fact that you can see the failure yourself puts you ahead of most people
using these tools.
</p>
</div>

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself — everyone must find one</h3>
<p style="color:#900; margin-bottom:0;">
Find a question your assistant <b>correctly refuses.</b> Not a silly one — something that
sounds like it might plausibly be in the documents but isn't.<br/><br/>
Then find a question where the refusal <b>fails</b>: it answers anyway, from general
knowledge, when it shouldn't.<br/><br/>
Bring both to Lesson 8. The second one is more interesting than the first.
</p>
</div>

In [ ]:
# Your turn

for question in ["", ""]:
    if not question:
        continue
    answer, sources = answer_question(question)
    print(f"Q: {question}\nA: {answer}\n   Sources: {', '.join(sources)}\n")

## PART D: The two dials

### Dial 1 — `k`

How many chunks you retrieve. Everyone's instinct is "more is better". It isn't.

In [ ]:
question = "What happened on Flight 5?"

for k in [1, 4, 12]:
    answer, sources = answer_question(question, k=k)
    print(f"===== k = {k}  (sources: {', '.join(sources)}) =====")
    print(answer)
    print()

**k too low** (1): if the one chunk you retrieved isn't the right one, you get a refusal
for a question that *is* answerable. Brittle.

**k too high** (12): you're back to Lesson 2's third wall. The right chunk is in there,
buried among eleven irrelevant ones, and answers get longer, vaguer and more expensive.

**4 to 6** is the usual starting point. As always — start there, then measure.

### Dial 2 — the system prompt

You've already seen this one do more work than any line of Python in this notebook.
Compare the two templates directly on the same question:

In [ ]:
question = "Who was on the team for Flight 8?"

print("=== LOOSE template ===")
print(answer_question(question, template=SYSTEM_TEMPLATE)[0])
print()
print("=== STRICT template ===")
print(answer_question(question, template=STRICT_TEMPLATE)[0])

## PART D: Conversation memory

Try this. Ask a question, then ask a follow-up that depends on it.

In [ ]:
print(answer_question("Why does every flight carry two trackers?")[0])
print()
print(answer_question("Which of them is more expensive?")[0])

The second answer is probably confused or a refusal — and it should be, because
`answer_question` has **no idea the first question happened.** Every call starts from
nothing.

Worse, the *retrieval* starts from nothing too. "Which of them is more expensive?" gets
embedded on its own, and "them" means nothing without the previous turn, so the search
brings back the wrong chunks before the model even sees anything.

That's two separate problems, and they need two different fixes.

### Fix 1: let the model see the conversation

Instead of sending one human message, send the whole exchange. The messages list simply
gets longer.

A conversation is just a list of `(question, answer)` pairs that we keep adding to — no
new concepts needed.

In [ ]:
from langchain_core.messages import AIMessage


def answer_with_memory(question, history, k=K):
    """
    Answer a question, taking the earlier conversation into account.

    history - a list of (question, answer) pairs from earlier turns

    Returns (answer, sources). The history is NOT re-retrieved - we only re-send it to
    the model so it knows what "them" refers to.
    """
    chunks = vectorstore.similarity_search(question, k=k)
    context = "\n\n---\n\n".join(doc.page_content for doc in chunks)

    messages = [SystemMessage(content=STRICT_TEMPLATE.format(context=context))]

    # Replay the conversation so far, in order, as alternating human/AI messages
    for past_question, past_answer in history:
        messages.append(HumanMessage(content=past_question))
        messages.append(AIMessage(content=past_answer))

    messages.append(HumanMessage(content=question))

    response = model.invoke(messages)
    return response.text, sorted({c.metadata["filename"] for c in chunks})

In [ ]:
# Now the follow-up has something to refer back to

history = []

q1 = "Why does every flight carry two trackers?"
a1, _ = answer_with_memory(q1, history)
print(f"Q: {q1}\nA: {a1}\n")
history.append((q1, a1))

q2 = "Which of them is more expensive?"
a2, _ = answer_with_memory(q2, history)
print(f"Q: {q2}\nA: {a2}")
history.append((q2, a2))

Better. The model now knows what "them" means.

But the retrieval is still searching for the bare words *"Which of them is more
expensive?"* — which, on its own, is about nothing at all.

### Fix 2: search using the conversation too

The cheap, surprisingly effective trick: glue the recent questions onto the current one
**just for the search**. The model still receives the real question; only the search
query gets the extra context.

There are cleverer approaches — you can ask a model to rewrite the follow-up into a
standalone question — but they cost an extra call every turn, and this gets most of the
benefit for free.

In [ ]:
def answer_with_memory(question, history, k=K):
    """
    Answer a question, taking the earlier conversation into account.

    Two separate uses of the history, and it's worth keeping them straight:
      - the SEARCH gets the recent questions glued on, so a follow-up like
        "which of them is cheaper?" retrieves the right chunks
      - the MODEL gets the real conversation replayed, so it knows what "them" means

    We only glue on the last two questions. More than that and the search query drifts
    toward whatever the conversation used to be about instead of what was just asked.
    """
    recent = [q for q, _ in history[-2:]]
    search_query = " ".join(recent + [question])

    chunks = vectorstore.similarity_search(search_query, k=k)
    context = "\n\n---\n\n".join(doc.page_content for doc in chunks)

    messages = [SystemMessage(content=STRICT_TEMPLATE.format(context=context))]
    for past_question, past_answer in history:
        messages.append(HumanMessage(content=past_question))
        messages.append(AIMessage(content=past_answer))
    messages.append(HumanMessage(content=question))

    response = model.invoke(messages)
    return response.text, sorted({c.metadata["filename"] for c in chunks})

In [ ]:
# A real three-turn conversation

history = []

for question in [
    "Why does every flight carry two trackers?",
    "Which of them is more expensive?",
    "Has it ever actually been needed?",
]:
    answer, sources = answer_with_memory(question, history)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"   sources: {', '.join(sources)}\n")
    history.append((question, answer))

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Have a <b>four-turn conversation</b> with your own documents where every question after
the first depends on the one before it.<br/><br/>
Then deliberately break it: change <code>history[-2:]</code> to <code>history[-8:]</code>
and have a conversation that <b>changes subject halfway through</b>. Watch the retrieval
get worse — it's still searching for what you were talking about five minutes ago.<br/><br/>
How much memory is too much? There's no correct answer, which is exactly why it's worth
finding out for yourself.
</p>
</div>

In [ ]:
# Your conversation

history = []

for question in ["", "", "", ""]:
    if not question:
        continue
    answer, sources = answer_with_memory(question, history)
    print(f"Q: {question}\nA: {answer}\n   sources: {', '.join(sources)}\n")
    history.append((question, answer))

## Where we got to

**You have built a working RAG system.** In full:

```
question
   ↓
similarity_search(question, k=4)          ← Lesson 4
   ↓
join the chunks into a context string     ← today
   ↓
paste into the system prompt              ← Lesson 2
   ↓
model.invoke([system, human])             ← Lesson 1
   ↓
answer + sources
```

- Read the assembled prompt. Almost every "why did it say that?" is answered there
- Return sources, so an answer can be checked rather than trusted
- The refusal sentence is what turns a confident liar into something useful
- `k` is a real trade-off in both directions; 4-6 to start
- Swap the model with one line — with Ollama the answering half even runs on your
  own laptop (searching stays on OpenAI either way, and costs a tiny fraction of a cent)

## Next lesson — the big one

**We leave the notebook.**

And there's a genuinely new idea in it: this splits into **two programs**. One that reads
your documents and builds the index (slow, run it when documents change), and one that
answers questions (fast, run it constantly).

Right now we rebuild the entire vector store every single time we open this notebook, which
is absurd. Next session we fix that properly.

Bring all five notebooks, and bring your own documents.